In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

import sys
import os
import pickle
from tqdm import tqdm
# Get the absolute path to the folder containing `utils`
utils_path = os.path.abspath('../')
if utils_path not in sys.path:
    sys.path.append(utils_path)
    
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # second gpu
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]="platform"
    
from utils import *

# az.style.use("arviz-docgrid")
plt.rcParams['figure.dpi'] = 140

experiment_orientations = [159, 123, 87, 51, 15]
subjects = ["01", "02", "03", "04", "05", "06", "07" ,"09", "10", "11", "12"]
median_key = {15:0, 51:1, 87:2, 123:3, 159:4}
std_key = {15:0, 51:1, 87:2, 123:3, 159:4}

c_table = pd.read_csv('../data_caches/ctiraltable.csv')
med = np.load('../data_caches/med.npy')
std = np.load('../data_caches/std.npy')
ctimetable = np.load('../data_caches/ctimetable.npy')
r_table = pd.read_csv('../data_caches/rtrialtable.csv', index_col=0)
(x, y, d, r, e, cd, ce) = np.load('../data_caches/rtimetable.npy', allow_pickle=True)

In [2]:
import jax
jax.config.update('jax_platform_name', 'gpu')

import jax.numpy as jnp
import jax.random as jr
from jax import lax
from jax import vmap
import optax

from jax.extend import backend
print(backend.get_backend().platform)

gpu


# Data

In [3]:
xaxis = np.arange(-250, 750, 1) * (1000/120)
start_idx, end_idx = np.searchsorted(xaxis, -500), np.searchsorted(xaxis, 1500)
(start_idx, end_idx)

(np.int64(191), np.int64(430))

In [4]:
# our frmes of intrest are only
er = e[:, :, :, :, start_idx:end_idx].reshape(-1, 239)

# ds
ds = d[:, :, :, :, start_idx:end_idx].reshape(-1, 239)

er[np.isnan(er)] = 90
ds[np.isnan(ds)] = 0

# min max scale err
er = er / 180

emissions = jnp.array(np.stack([ds, er], axis=-1))

subset_idx = np.random.choice(np.arange(0, len(emissions)), 400, replace=False)
sub_em = emissions[subset_idx]

er.shape, np.isnan(er).any(), ds.shape, np.isnan(ds).any(), emissions.shape

((34560, 239), np.False_, (34560, 239), np.False_, (34560, 239, 2))

# Models

In [5]:
num_states, emission_dim = 3, 2

## Global initializer

In [6]:
emissions.shape

(34560, 239, 2)

In [7]:
# global_params = {}
# global_lps = {}

# for nstate in range(2, 10):
#     model = GaussianHMM(nstate, emission_dim, transition_matrix_stickiness=10.)
#     parameters, properties = model.initialize(key=jr.PRNGKey(1), method="prior")
#     subset_idx = np.random.choice(np.arange(0, len(emissions)), len(emissions), replace=False)
#     fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = emissions[subset_idx], num_iters = 10, verbose = True)
#     global_params[nstate] = fit_params
#     global_lps[nstate] = lps

In [8]:
# with open('./caches/global_params.pkl', 'wb') as f:
#     pickle.dump(global_params, f)

with open('./caches/global_params.pkl', 'rb') as f:
    global_params = pickle.load(f)

## Crossvalidate subject models LOO

In [9]:
# model = GaussianHMM(3, 2, transition_matrix_stickiness=10.)
# parameters, properties = model.initialize(key=jr.PRNGKey(1),  method="prior")
# ll_mean, ll = cross_validate_model(model=model, emissions=emissions[:100], key=jr.PRNGKey(0), num_iters=10, init = (parameters, properties))

In [10]:
# model = GaussianHMM(2, emission_dim)
# parameters, properties = model.initialize(key=jr.PRNGKey(1), method="prior")
# fit_params, ll = model.fit_em(parameters, properties, emissions[:2880], num_iters=100, verbose=True)
# plt.plot(ll)

In [11]:
# crossval_results = {}
# sd = emissions.reshape(12, 4 * 6 * 120, 239, 2)

# for idx in range(0, 12):
#     crossval_results[idx] = {}
    
#     subset_idx = np.random.choice(np.arange(0, 2880), 300, replace=False)
#     ss_em = sd[idx][subset_idx]
    
#     for nstate in tqdm(range(2, 10), desc=f'sub-{idx+1}'):
#         model = GaussianHMM(nstate, emission_dim)
#         gpar = global_params[nstate]
#         parameters, properties = model.initialize(key=jr.PRNGKey(1), 
#                                                   method="prior",
#                                                   initial_probs=gpar.initial.probs,
#                                                   transition_matrix=gpar.transitions.transition_matrix,
#                                                   emission_means=gpar.emissions.means,
#                                                   emission_covariances=gpar.emissions.covs,)
#         ll_mean, ll = cross_validate_dist(model=model, emissions=ss_em, key=jr.PRNGKey(0), num_iters=10, init = (parameters, properties), num_folds=100)
#         # ll_mean, ll = cross_validate_dist(model=model, emissions=ss_em, key=jr.PRNGKey(0), num_iters=10, init = "default", num_folds=100)
#         crossval_results[idx][nstate] = ll

In [12]:
# with open('./caches/crossval_results.pkl', 'wb') as f:
#     pickle.dump(crossval_results, f)
    
# with open('./caches/crossval_results.pkl', 'rb') as f:
#     crossval_results = pickle.load(f)

## Init fitting Subject Models for 3 state 

In [13]:
emissions.shape

(34560, 239, 2)

In [14]:
# subject_params = {}

# for idx, s_em in enumerate(emissions.reshape(12, 4 * 6 * 120, 239, 2)):
#     shuffle_idx = np.random.permutation(len(s_em))
    
#     nstate = 2
#     model = GaussianHMM(nstate, emission_dim)
#     gpar = global_params[nstate]
#     parameters, properties = model.initialize(key=jr.PRNGKey(1), 
#                                                 method="prior",
#                                                 initial_probs=gpar.initial.probs,
#                                                 transition_matrix=gpar.transitions.transition_matrix,
#                                                 emission_means=gpar.emissions.means,
#                                                 emission_covariances=gpar.emissions.covs,)
#     fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = s_em[shuffle_idx], num_iters = 100, verbose = True)
    
#     subject_params[idx] = fit_params

In [15]:
# with open('./caches/subject_params.pkl', 'wb') as f:
#     pickle.dump(subject_params, f)
    
# with open('./caches/subject_params.pkl', 'rb') as f:
#     subject_params = pickle.load(f)

## Condition Fitting

In [ ]:
# cond_params = {}
# nstate = 2

# for val, cond in zip(["low", "high", "divided", "focused", "unexpected", "expected"], ["coh", "coh", "att", "att", "exp_al", "exp_al"]):
#     cond_params[val] = {}
    
#     for idx in tqdm(range(12), desc=f'{val}'):    
#         data_mask = (r_table["subj"] == idx+1) & (r_table[cond] == val)
#         shuffle_idx = np.random.permutation(sum(data_mask))
#         em = jnp.array(np.array(emissions)[data_mask][shuffle_idx])
#         model = GaussianHMM(nstate, emission_dim)
#         # gpar = global_params[nstate]
#         parameters, properties = model.initialize(key=jr.PRNGKey(1), 
#                                                     method="prior",
#                                                     initial_probs=gpar.initial.probs,
#                                                     transition_matrix=gpar.transitions.transition_matrix,
#                                                     emission_means=gpar.emissions.means,
#                                                     emission_covariances=gpar.emissions.covs,)
#         fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = em, num_iters = 100, verbose = True)
        
#         cond_params[val][idx] = fit_params

In [17]:
# with open('./caches/cond_params.pkl', 'wb') as f:
#     pickle.dump(cond_params, f)
    
with open('./caches/cond_params.pkl', 'rb') as f:
    cond_params = pickle.load(f)

## ML States

In [22]:
# ml_states = {}
# nstate = 2

# for val, cond in zip(["low", "high", "divided", "focused", "unexpected", "expected"], ["coh", "coh", "att", "att", "exp_al", "exp_al"]):
#     ml_states[val] = {}
    
#     for idx in tqdm(range(12), desc=f'{val}'):    
#         data_mask = (r_table["subj"] == idx+1) & (r_table[cond] == val)
#         shuffle_idx = np.random.permutation(sum(data_mask))
#         em = jnp.array(np.array(emissions)[data_mask][shuffle_idx])
#         model = GaussianHMM(num_states = nstate, emission_dim = 2)
#         gpar = cond_params[val][idx]
#         parameters, properties = model.initialize(key=jr.PRNGKey(1), 
#                                                     method="prior",
#                                                     initial_probs=gpar.initial.probs,
#                                                     transition_matrix=gpar.transitions.transition_matrix,
#                                                     emission_means=gpar.emissions.means,
#                                                     emission_covariances=gpar.emissions.covs,)
        
#         # Define a vmapped version of most_likely_states.
#         # This applies the function to each trial in the batch dimension of `em`.
#         most_likely_states_vmap = vmap(lambda trial: model.most_likely_states(parameters, trial))
#         t_trail = most_likely_states_vmap(em)
        
#         ml_states[val][idx] = np.array(t_trail)

In [ ]:
# with open('./caches/ml_states.pkl', 'wb') as f:
#     pickle.dump(ml_states, f)
    
with open('./caches/ml_states.pkl', 'rb') as f:
    ml_states = pickle.load(f)